# Experiência do usuário e QoE do dump Argus

## Escopo e atraso conhecido

Estas tabelas NPAW/Youbora são agregadas por hora x dimensão e não expõem IDs
individuais. Cada consulta usa uma janela UTC explícita `[start, end)`. Há um
atraso estrutural de aproximadamente **28 horas**: a cobertura mais recente pode
estar vazia ou parcial por desenho do pipeline, e não é evidência de ausência de
incidente em tempo real.

In [ ]:
from dump_explorer.charts import (
    category_stack,
    geo_heatmap,
    impact_scatter,
    qoe_timeline,
)
from dump_explorer.config import load_settings
from dump_explorer.database import create_db_engine
from dump_explorer.qoe import (
    NPAW_DELAY_HOURS,
    load_device_qoe,
    load_error_categories,
    load_geo_qoe,
    load_live_active,
    load_live_qoe,
    load_player_qoe,
)

# derive a safe UTC epoch-ms window from signal_bucket observed bounds
from dump_explorer.quality import load_source_coverage
import pandas as pd

settings = load_settings()
engine = create_db_engine(settings)

# Obter janela segura a partir da cobertura de dados
s = load_source_coverage(engine)
if s.empty or s['first_ts'].dropna().empty:
    print("⚠️  source_coverage vazio. Usando valores padrão.")
    start_ms = 0
    end_ms = 86_400_000
else:
    start_ms = int(s['first_ts'].dropna().min().value // 10**6)
    end_ms = int(s['last_ts'].dropna().max().value // 10**6) + 1
    print(f"Intervalo de dados: {pd.to_datetime(start_ms, unit='ms', utc=True)} a {pd.to_datetime(end_ms-1, unit='ms', utc=True)}")

## 1. Players, dispositivos e geografia

`(desconhecido)` é uma dimensão ausente explicitamente marcada, não removida do
ranking. A taxa de erro é `NULL` quando sessões é zero. Dispositivos usam pelo
menos 30 sessões **na consulta SQL**; o ranking não representa a cardinalidade
bruta. O heatmap UF x ISP usa somente agregados locais, sem mapa ou ativos externos.

In [ ]:
player_qoe = load_player_qoe(engine, start_ms, end_ms)
device_qoe = load_device_qoe(engine, start_ms, end_ms, min_sessions=30)
geo_qoe = load_geo_qoe(engine, start_ms, end_ms)
display(player_qoe)
display(device_qoe)
display(geo_qoe)
impact_scatter(player_qoe, dimension="player").show()
impact_scatter(device_qoe, dimension="device").show()
geo_heatmap(geo_qoe).show()

## 2. QoE ao vivo e audiência

Selecione sinais explicitamente. `bufferratio` é **ratio**, `jointime` é em
**segundos** e latência é em **milissegundos**; as unidades não compartilham eixo.
`startup_errors` fica separado de erros durante a reprodução. `active_sessions`
é uma contagem de sessões **concorrentes**, não acumulada; `joins_in_bucket`
significa novas entradas no bucket de 15 minutos.

In [ ]:
signals = ["pfc1"]  # selecione explicitamente um ou mais sinais

if not signals:
    print("⚠️  signals está vazia. Adicione nomes de sinais para visualizar dados QoE ao vivo.")
else:
    live_qoe = load_live_qoe(engine, signals, start_ms, end_ms)
    live_active = load_live_active(engine, signals, start_ms, end_ms)
    latest_qoe_timestamp = live_active["timestamp"].max() if not live_active.empty else None
    
    if latest_qoe_timestamp:
        print(
            f"Timestamp NPAW mais recente: {latest_qoe_timestamp}; "
            f"cobertura pode atrasar cerca de {NPAW_DELAY_HOURS} horas."
        )
    else:
        print(f"⚠️  Nenhum dado NPAW encontrado. Cobertura pode estar atrasada (~{NPAW_DELAY_HOURS} horas).")
    
    if not live_qoe.empty:
        display(live_qoe)
        try:
            qoe_timeline(live_qoe, metric="startup_errors").show()
            qoe_timeline(live_qoe, metric="bufferratio_avg").show()
            qoe_timeline(live_qoe, metric="jointime_s_avg").show()
        except Exception as e:
            print(f"❌ Erro ao renderizar timelines QoE: {e}")
    else:
        print(f"⚠️  Nenhum dado QoE encontrado para os sinais: {signals}")
    
    if not live_active.empty:
        display(live_active)
        try:
            qoe_timeline(live_active, metric="active_sessions").show()
        except Exception as e:
            print(f"❌ Erro ao renderizar timeline de sessões ativas: {e}")
    else:
        print(f"⚠️  Nenhum dado de sessões ativas encontrado para os sinais: {signals}")

## 3. Categorias semânticas de erro

Live e VOD são carregados e visualizados separadamente; não existe taxa única
misturando os dois. `top_names` preserva o texto bruto e recebe uma flag quando
o JSON está inválido.

In [ ]:
live_categories = load_error_categories(engine, "live", start_ms, end_ms)
vod_categories = load_error_categories(engine, "vod", start_ms, end_ms)
display(live_categories)
display(vod_categories)
category_stack(live_categories).show()
category_stack(vod_categories).show()